In [ ]:
# Chapter 16: Natural Language Processing with RNNs and Attention

## Global Imports

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import tensorflow_datasets as tfds

## Generating Shakespearean Text Using a Character RNN
The chapter begins by tackling a fun task: training an RNN to predict the next character in a sentence, essentially teaching it to write like Shakespeare. This is known as a Character RNN (Char-RNN).

### Creating the Dataset
First, we download all of Shakespeare's work and encode the text so each character is represented by an integer ID. We use a tokenizer to fit on the text.

In [2]:
shakespeare_url = "https://homl.info/shakespeare"
filepath = keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
    shakespeare_text = f.read()

tokenizer = keras.preprocessing.text.Tokenizer(char_level=True)
tokenizer.fit_on_texts([shakespeare_text])

# Encode the text
[encoded] = np.array(tokenizer.texts_to_sequences([shakespeare_text])) - 1
max_id = len(tokenizer.word_index)
dataset_size = tokenizer.document_count

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


### Splitting a Sequential Dataset
We cannot train on the entire text at once. We split the time series into windows. We use a window dataset to create small windows of text (e.g., 101 characters). The target is the character immediately following the input window (input: 0-100, target: 1-101).

<p align="left"><img src="../fig/figure16.1.png" width="45%"></p>

In [3]:
train_size = dataset_size * 90 // 100
dataset = tf.data.Dataset.from_tensor_slices(encoded[:train_size])

n_steps = 100
window_length = n_steps + 1 # target = input shifted by 1
dataset = dataset.window(window_length, shift=1, drop_remainder=True)
dataset = dataset.flat_map(lambda window: window.batch(window_length))

batch_size = 32
dataset = dataset.shuffle(10000).batch(batch_size)
dataset = dataset.map(lambda windows: (windows[:, :-1], windows[:, 1:]))
dataset = dataset.map(
    lambda X_batch, Y_batch: (tf.one_hot(X_batch, depth=max_id), Y_batch))
dataset = dataset.prefetch(1)

### Building and Training the Char-RNN Model
We use a model with two GRU layers. Note that we do not use return_sequences=True for the second layer because we are using a Dense layer applied to every time step, but in this specific architecture shown in the book for stateless training on many-to-many, we often configure it carefully. However, for predicting the next character at every step, the model looks like this:

In [4]:
model = keras.models.Sequential([
    keras.layers.GRU(128, return_sequences=True, input_shape=[None, max_id],
                     dropout=0.2), # dropout is deprecated in new TF, use recurrent_dropout or separate
    keras.layers.GRU(128, return_sequences=True,
                     dropout=0.2),
    keras.layers.TimeDistributed(keras.layers.Dense(max_id,
                                                    activation="softmax"))
])
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")
# history = model.fit(dataset, epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


### Generating Text
To generate text, we feed the model a seed text, predict the next character (by sampling from the output probabilities using tf.random.categorical to introduce diversity via a temperature parameter), append it to the text, and repeat.

In [5]:
def preprocess(texts):
    X = np.array(tokenizer.texts_to_sequences(texts)) - 1
    return tf.one_hot(X, max_id)

def next_char(text, temperature=1):
    X_new = preprocess([text])
    y_proba = model.predict(X_new)[0, -1:, :]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1) + 1
    return tokenizer.sequences_to_texts(char_id.numpy())[0]

# print(next_char("To be or not to b", temperature=1))

## Stateful RNN
In a standard (Stateless) RNN, the hidden state is reset to zero at the beginning of each batch. This prevents the model from learning long-term patterns that cross batch boundaries. In a Stateful RNN, the final state of one batch is used as the initial state for the next batch.
To implement this, we must:
1. Use sequential (non-shuffled) batches.
2. Set stateful=True in the RNN layers.
3. Specify batch_input_shape.
4. Manually reset states at the end of each epoch.

<p align="left"><img src="../fig/figure16.2.png" width="45%"></p>

In [7]:
import tensorflow as tf
from tensorflow import keras

# Asumsi variabel batch_size dan max_id sudah didefinisikan sebelumnya
# batch_size = 32
# max_id = ... (sesuai vocab size)

# Create a stateful model
model = keras.models.Sequential([
    # PERBAIKAN 1: Gunakan layer Input dengan 'batch_shape' untuk stateful RNN
    keras.layers.Input(batch_shape=[batch_size, None, max_id]),

    # Hapus batch_input_shape dari argument GRU
    keras.layers.GRU(128, return_sequences=True, stateful=True, dropout=0.2),

    keras.layers.GRU(128, return_sequences=True, stateful=True, dropout=0.2),

    keras.layers.TimeDistributed(keras.layers.Dense(max_id, activation="softmax"))
])

# Callback to reset states
class ResetStatesCallback(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs):
        # PERBAIKAN 2: Gunakan reset_state() (tunggal) bukan reset_states()
        self.model.reset_state()

model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")

# Pastikan dataset sudah siap sebelum menjalankan fit
# model.fit(dataset, epochs=50, callbacks=[ResetStatesCallback()])

## Sentiment Analysis
We can use RNNs for classification tasks, such as determining if a movie review is positive or negative using the IMDB dataset.

### Loading and Preprocessing
We use tfds to load the dataset. We need to preprocess the text (truncate/pad to same length) and encode words.

In [8]:
dataset, info = tfds.load("imdb_reviews", as_supervised=True, with_info=True)
train_size = info.splits["train"].num_examples

def preprocess(X_batch, y_batch):
    X_batch = tf.strings.substr(X_batch, 0, 300) # Truncate for speed
    X_batch = tf.strings.regex_replace(X_batch, b"<br\\s*/?>", b" ")
    X_batch = tf.strings.regex_replace(X_batch, b"[^a-zA-Z']", b" ")
    X_batch = tf.strings.split(X_batch)
    return X_batch.to_tensor(default_value=b"<pad>"), y_batch

# Vocabulary construction logic (simplified) would go here
from collections import Counter
vocabulary = Counter()
for X_batch, y_batch in dataset["train"].batch(32).map(preprocess):
    for review in X_batch:
        vocabulary.update(list(review.numpy()))

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.8AUVPG_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.8AUVPG_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.8AUVPG_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


### Masking
When padding sequences, it's crucial to tell the model to ignore padding tokens. Keras Embedding layer supports mask_zero=True.

In [9]:
vocab_size = 10000
embed_size = 128

model = keras.models.Sequential([
    # mask_zero=True creates a mask tensor propagated to next layers
    keras.layers.Embedding(vocab_size + 1, embed_size,
                           mask_zero=True, input_shape=[None]),
    keras.layers.GRU(128, return_sequences=True),
    keras.layers.GRU(128),
    keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


## Encoder-Decoder Network for Neural Machine Translation (NMT)
For tasks like translation (e.g., English to Spanish), we use an Encoder-Decoder architecture.
- Encoder: An RNN that reads the input sentence and outputs a final hidden state (context vector).
- Decoder: An RNN that takes the context vector and generates the translation step-by-step.
We use TensorFlow Addons (tfa) for specialized seq2seq components (note: in newer TF versions, this is often handled differently, but the book refers to tfa or custom loops). The snippet below shows a basic custom implementation using Keras layers.

<p align="left"><img src="../fig/figure16.3.png" width="45%"></p>
<p align="left"><img src="../fig/figure16.4.png" width="45%"></p>

In [10]:
embedding_size = 32
max_length = 50 # Example length

model = keras.models.Sequential()
model.add(keras.layers.Embedding(vocab_size, embedding_size,
                                 input_length=max_length, mask_zero=True))
model.add(keras.layers.LSTM(128)) # Encoder
model.add(keras.layers.RepeatVector(max_length)) # Repeat context for decoder
model.add(keras.layers.LSTM(128, return_sequences=True)) # Decoder
model.add(keras.layers.Dense(vocab_size, activation="softmax"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## Attention Mechanisms
The basic Encoder-Decoder has a bottleneck: a single fixed-size vector must hold the meaning of the entire sentence. Attention mechanisms allow the decoder to focus on specific parts of the input sentence at each time step.
- Bahdanau Attention (Additive): Computes alignment scores using a feed-forward network.
- Luong Attention (Multiplicative): Computes alignment scores using dot products.
Implementing Attention in Keras usually involves subclassing or using tf.keras.layers.Attention (Luong) or AdditiveAttention (Bahdanau).

In [11]:
# Example logic using Keras built-in layers for attention
# Encoder
encoder_inputs = keras.layers.Input(shape=(None,), dtype="int32")
encoder_embedding = keras.layers.Embedding(vocab_size, embed_size)(encoder_inputs)
encoder_outputs, state_h, state_c = keras.layers.LSTM(128, return_state=True)(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = keras.layers.Input(shape=(None,), dtype="int32")
decoder_embedding = keras.layers.Embedding(vocab_size, embed_size)(decoder_inputs)
decoder_lstm = keras.layers.LSTM(128, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

# Attention Layer (Luong-style)
attention_layer = keras.layers.Attention()
attention_outputs = attention_layer([decoder_outputs, encoder_outputs])

# Concatenate and Output
concat_layer = keras.layers.Concatenate()
decoder_concat_input = concat_layer([decoder_outputs, attention_outputs])
output_layer = keras.layers.TimeDistributed(keras.layers.Dense(vocab_size, activation="softmax"))
decoder_final_outputs = output_layer(decoder_concat_input)

## The Transformer Architecture
The Transformer, introduced in the paper "Attention Is All You Need" (2017), eliminates RNNs entirely. It relies solely on attention mechanisms, making it highly parallelizable and effective for long sequences.

### Architecture Components
1. Positional Encoding: Since there are no recurrent steps, the model needs information about the position of words. Sine and cosine functions of different frequencies are added to the input embeddings.
2. Multi-Head Attention: Several attention layers run in parallel. Each "head" can focus on different subspaces of the relationship (e.g., one head focuses on grammar, another on semantic relationship).
3. Self-Attention: The encoder attends to itself (each word looks at every other word in the same sentence) to understand context.

### Building a Simplified Transformer Block
A Transformer block consists of Multi-Head Attention followed by a Feed-Forward Network, with Skip Connections and Layer Normalization.

In [12]:
class TransformerBlock(keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.models.Sequential(
            [keras.layers.Dense(ff_dim, activation="relu"), keras.layers.Dense(embed_dim),]
        )
        self.layernorm1 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = keras.layers.Dropout(rate)
        self.dropout2 = keras.layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

This architecture is the foundation of modern NLP models like BERT (Bidirectional Encoder Representations from Transformers) and GPT (Generative Pre-trained Transformer).